# Create custom log handlers

**Problem:** You need to send log messages to a destination that is not covered
by the built-in handlers, such as an in-memory buffer, a CSV file, or an external
notification system.

This guide shows you how to create custom handlers by subclassing `logging.Handler`.

## How handlers work

Every handler in the `logging` module inherits from `logging.Handler`. The key
method to override is `emit(record)`, which receives a `logging.LogRecord` object
and is responsible for outputting the log message.

The base class provides:

- `setLevel()` and `setFormatter()` for configuration
- `format(record)` to apply the formatter and produce a string
- `handleError(record)` for error handling within the handler itself

## Creating a simple custom handler

Let us start with a `ListHandler` that stores log messages in a Python list.
This is useful for testing and for capturing log output programmatically.

In [ ]:
import logging


class ListHandler(logging.Handler):
    """A handler that stores formatted log messages in a list.

    This is useful for testing or for capturing log output programmatically.

    Attributes:
        messages: A list of formatted log message strings.
    """

    def __init__(self, level: int = logging.NOTSET) -> None:
        """Initialise the handler.

        Args:
            level: The minimum log level to capture.
        """
        super().__init__(level)
        self.messages: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        """Store the formatted log record.

        Args:
            record: The log record to store.
        """
        message = self.format(record)
        self.messages.append(message)


# Use the custom handler
logger = logging.getLogger("list_handler_demo")
logger.setLevel(logging.DEBUG)

handler = ListHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

logger.info("First message")
logger.warning("Second message")
logger.error("Third message")

print("Captured messages:")
for msg in handler.messages:
    print(" ", msg)

logger.removeHandler(handler)

## A practical example: CSV handler

Here is a more practical example: a handler that writes log records to a CSV file.
Each row contains the timestamp, level, logger name, and message.

In [ ]:
import csv
import logging
import os
import tempfile
from datetime import datetime, timezone


class CSVHandler(logging.Handler):
    """A handler that writes log records to a CSV file.

    Each row contains the timestamp, level, logger name, and message.
    """

    def __init__(self, filepath: str, level: int = logging.NOTSET) -> None:
        """Initialise the CSV handler.

        Args:
            filepath: Path to the CSV file.
            level: The minimum log level to capture.
        """
        super().__init__(level)
        self.filepath = filepath
        # Write the header row
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", "level", "logger", "message"])

    def emit(self, record: logging.LogRecord) -> None:
        """Write a log record as a CSV row.

        Args:
            record: The log record to write.
        """
        try:
            timestamp = datetime.fromtimestamp(
                record.created, tz=timezone.utc
            ).strftime("%d/%m/%Y %H:%M:%S")
            with open(self.filepath, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    timestamp,
                    record.levelname,
                    record.name,
                    record.getMessage(),
                ])
        except Exception:
            self.handleError(record)


# Use the CSV handler
log_dir = tempfile.mkdtemp()
csv_path = os.path.join(log_dir, "logs.csv")

logger = logging.getLogger("csv_demo")
logger.setLevel(logging.DEBUG)

csv_handler = CSVHandler(csv_path)
logger.addHandler(csv_handler)

logger.info("Application started")
logger.warning("Disk space low")
logger.error("Connection failed")

print("CSV file contents:")
with open(csv_path, "r", encoding="utf-8") as f:
    print(f.read())

logger.removeHandler(csv_handler)

## Adding filtering to custom handlers

You can add filters to any handler using `addFilter()`. Filters let you accept
or reject log records based on custom criteria.

In [ ]:
import logging


class KeywordFilter(logging.Filter):
    """A filter that only accepts records containing a specific keyword.

    Attributes:
        keyword: The keyword to search for in the log message.
    """

    def __init__(self, keyword: str) -> None:
        """Initialise the filter.

        Args:
            keyword: The keyword to filter on.
        """
        super().__init__()
        self.keyword = keyword

    def filter(self, record: logging.LogRecord) -> bool:
        """Return True if the record contains the keyword.

        Args:
            record: The log record to check.

        Returns:
            True if the keyword is found in the message.
        """
        return self.keyword in record.getMessage()


# Use the filter with a handler
logger = logging.getLogger("filter_demo")
logger.setLevel(logging.DEBUG)

handler = ListHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
handler.addFilter(KeywordFilter("security"))
logger.addHandler(handler)

logger.info("User logged in")
logger.warning("security: Failed login attempt")
logger.info("Data processed")
logger.error("security: Unauthorised access attempt")

print("Only security-related messages captured:")
for msg in handler.messages:
    print(" ", msg)

logger.removeHandler(handler)

## Error handling in custom handlers

It is important to handle errors gracefully in the `emit()` method. If your
handler fails (for example, a network error or file permission issue), calling
`self.handleError(record)` ensures the error is reported without crashing the
application.

Always wrap the body of `emit()` in a `try`/`except` block and call
`self.handleError(record)` in the `except` clause, as shown in the CSV handler
example above.

## Testing custom handlers

You can test custom handlers using `unittest`. The `ListHandler` shown earlier
is itself a useful testing tool -- you can attach it to a logger and inspect
the captured messages.

In [ ]:
import logging
import unittest


class TestListHandler(unittest.TestCase):
    """Tests for the ListHandler class."""

    def test_captures_messages(self) -> None:
        """Test that messages are captured in the list."""
        logger = logging.getLogger("test_list_handler")
        logger.setLevel(logging.DEBUG)
        handler = ListHandler()
        handler.setFormatter(logging.Formatter("%(message)s"))
        logger.addHandler(handler)

        logger.info("Test message")

        self.assertEqual(len(handler.messages), 1)
        self.assertIn("Test message", handler.messages[0])

        logger.removeHandler(handler)

    def test_respects_level(self) -> None:
        """Test that messages below the handler level are not captured."""
        logger = logging.getLogger("test_list_handler_level")
        logger.setLevel(logging.DEBUG)
        handler = ListHandler(level=logging.WARNING)
        logger.addHandler(handler)

        logger.debug("Debug message")
        logger.warning("Warning message")

        self.assertEqual(len(handler.messages), 1)

        logger.removeHandler(handler)


# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestListHandler)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Summary

To create a custom log handler:

1. Subclass `logging.Handler`
2. Override the `emit(record)` method
3. Use `self.format(record)` to get the formatted message string
4. Wrap the `emit()` body in `try`/`except` and call `self.handleError(record)` on failure
5. Add filters with `addFilter()` for fine-grained control
6. Test your handler with `unittest`